# 🛰️ 01 — Знакомство с данными MARIDA

Цель: понять структуру датасета, что хранится в файлах, как устроены патчи и метки.

**Датасет:** MARIDA (Marine Debris Archive) — снимки Sentinel-2 с размеченными объектами на поверхности океана.

**Задача:** классифицировать каждый пиксель на 4 класса — Мусор / Водоросли / Пена / Вода.

In [1]:
import numpy as np
import rasterio
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

In [3]:
DATA_PATH    = Path(r"c:\Users\Amir\Desktop\EcoHack\data")
PATCHES_PATH = DATA_PATH / "patches"
SPLITS_PATH  = DATA_PATH / "splits"

print("DATA_PATH существует:", DATA_PATH.exists())
print("Содержимое:", [p.name for p in DATA_PATH.iterdir()])

DATA_PATH существует: True
Содержимое: ['labels_mapping.txt', 'patches', 'shapefiles', 'splits']


## 1. Структура директории с патчами

Каждый патч — это отдельная папка внутри `data/patches/`. Посмотрим сколько сцен и что внутри одной из них.

In [4]:
scenes = sorted(PATCHES_PATH.iterdir())
print(f"Всего сцен: {len(scenes)}")
print("\nПервые 5 сцен:")
for s in scenes[:5]:
    print(f"  {s.name}")

Всего сцен: 63

Первые 5 сцен:
  S2_1-12-19_48MYU
  S2_11-1-19_19QDA
  S2_11-6-18_16PCC
  S2_12-1-17_16PCC
  S2_12-1-17_16PEC


In [5]:
example_scene = scenes[0]
files_in_scene = sorted(example_scene.iterdir())
print(f"Сцена: {example_scene.name}")
print(f"Файлов: {len(files_in_scene)}")
print("\nВсе файлы:")
for f in files_in_scene:
    print(f"  {f.name}  [{f.stat().st_size // 1024} КБ]")

Сцена: S2_1-12-19_48MYU
Файлов: 12

Все файлы:
  S2_1-12-19_48MYU_0.tif  [2817 КБ]
  S2_1-12-19_48MYU_0_cl.tif  [256 КБ]
  S2_1-12-19_48MYU_0_conf.tif  [256 КБ]
  S2_1-12-19_48MYU_1.tif  [2817 КБ]
  S2_1-12-19_48MYU_1_cl.tif  [256 КБ]
  S2_1-12-19_48MYU_1_conf.tif  [256 КБ]
  S2_1-12-19_48MYU_2.tif  [2817 КБ]
  S2_1-12-19_48MYU_2_cl.tif  [256 КБ]
  S2_1-12-19_48MYU_2_conf.tif  [256 КБ]
  S2_1-12-19_48MYU_3.tif  [2817 КБ]
  S2_1-12-19_48MYU_3_cl.tif  [256 КБ]
  S2_1-12-19_48MYU_3_conf.tif  [256 КБ]


In [6]:
suffixes = Counter()
for scene in scenes:
    for f in scene.iterdir():
        name = f.name
        if name.endswith("_cl.tif"):
            suffixes["_cl.tif (маска классов)"] += 1
        elif name.endswith("_conf.tif"):
            suffixes["_conf.tif (уверенность)"] += 1
        elif name.endswith(".tif"):
            suffixes[".tif (снимок)"] += 1

print("Типы файлов в патчах:")
for k, v in suffixes.items():
    print(f"  {k:35s} — {v} файлов")

Типы файлов в патчах:
  .tif (снимок)                       — 1381 файлов
  _cl.tif (маска классов)             — 1381 файлов
  _conf.tif (уверенность)             — 1381 файлов


## 2. Схема меток — классы MARIDA

В файлах `_cl.tif` хранятся маски с исходными метками датасета MARIDA. Каждый пиксель имеет одно из 12 значений (0–11).

In [7]:
# 11 классов датасета MARIDA
CLASS_NAMES = {
    0:  "Нет данных",
    1:  "Морской мусор",
    2:  "Плотные саргассы",
    3:  "Редкие саргассы",
    4:  "Природный органический материал",
    5:  "Корабль",
    6:  "Облака",
    7:  "Морская вода",
    8:  "Морская пена",
    9:  "Мутная вода",
    10: "Мелководье",
    11: "Волны",
}

print("Классы MARIDA:")
for k, v in CLASS_NAMES.items():
    print(f"  {k:2d}  {v}")

Классы MARIDA:
   0  Нет данных
   1  Морской мусор
   2  Плотные саргассы
   3  Редкие саргассы
   4  Природный органический материал
   5  Корабль
   6  Облака
   7  Морская вода
   8  Морская пена
   9  Мутная вода
  10  Мелководье
  11  Волны


In [8]:
LABEL_REMAP = {
    0:  None,  # Нет данных
    1:  0,     # Морской мусор             → Мусор
    2:  1,     # Плотные саргассы          → Водоросли
    3:  1,     # Редкие саргассы           → Водоросли
    4:  1,     # Природный орг. материал   → Водоросли
    5:  None,  # Корабль                   → пропуск
    6:  None,  # Облака                    → пропуск
    7:  3,     # Морская вода              → Вода
    8:  2,     # Морская пена              → Пена
    9:  3,     # Мутная вода               → Вода
    10: 3,     # Мелководье                → Вода
    11: 2,     # Волны                     → Пена
}

TARGET_NAMES = {0: "Мусор", 1: "Водоросли", 2: "Пена", 3: "Вода"}

print("Переотображение в 4 класса:")
print(f"  {'MARIDA класс':<40s}  →  Целевой класс")
print("-" * 60)
for src, tgt in LABEL_REMAP.items():
    tgt_str = TARGET_NAMES[tgt] if tgt is not None else "(пропуск)"
    print(f"  {src:2d}  {CLASS_NAMES[src]:<35s}  →  {tgt_str}")

Переотображение в 4 класса:
  MARIDA класс                              →  Целевой класс
------------------------------------------------------------
   0  Нет данных                           →  (пропуск)
   1  Морской мусор                        →  Мусор
   2  Плотные саргассы                     →  Водоросли
   3  Редкие саргассы                      →  Водоросли
   4  Природный органический материал      →  Водоросли
   5  Корабль                              →  (пропуск)
   6  Облака                               →  (пропуск)
   7  Морская вода                         →  Вода
   8  Морская пена                         →  Пена
   9  Мутная вода                          →  Вода
  10  Мелководье                           →  Вода
  11  Волны                                →  Пена


## 3. Каналы Sentinel-2

MARIDA использует 11 мультиспектральных каналов Sentinel-2. Каждый канал соответствует определённому диапазону длин волн.

In [9]:
BAND_NAMES = ["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B11","B12"]
BAND_WL    = [443,   490,  560,  665,  705,  740,  783,  842,  865,  1610, 2190]
BAND_DESC  = [
    "Аэрозоль (Coastal Aerosol)",
    "Синий (Blue)",
    "Зелёный (Green)",
    "Красный (Red)",
    "Красный край 1 (Red Edge 1)",
    "Красный край 2 (Red Edge 2)",
    "Красный край 3 (Red Edge 3)",
    "БИК широкий (NIR Broad)",
    "БИК узкий (NIR Narrow)",
    "SWIR 1",
    "SWIR 2",
]

print(f"{'Индекс':<8} {'Имя':<6} {'λ (нм)':<8} Описание")
print("-" * 60)
for i, (name, wl, desc) in enumerate(zip(BAND_NAMES, BAND_WL, BAND_DESC)):
    print(f"  [{i:2d}]   {name:<6} {wl:<8} {desc}")

Индекс   Имя    λ (нм)   Описание
------------------------------------------------------------
  [ 0]   B01    443      Аэрозоль (Coastal Aerosol)
  [ 1]   B02    490      Синий (Blue)
  [ 2]   B03    560      Зелёный (Green)
  [ 3]   B04    665      Красный (Red)
  [ 4]   B05    705      Красный край 1 (Red Edge 1)
  [ 5]   B06    740      Красный край 2 (Red Edge 2)
  [ 6]   B07    783      Красный край 3 (Red Edge 3)
  [ 7]   B08    842      БИК широкий (NIR Broad)
  [ 8]   B8A    865      БИК узкий (NIR Narrow)
  [ 9]   B11    1610     SWIR 1
  [10]   B12    2190     SWIR 2


## 4. Разбивка на train / val / test

В папке `data/splits/` хранятся текстовые файлы с перечнем id патчей для каждого раздела.

In [10]:
split_files = sorted(SPLITS_PATH.iterdir())
print("Файлы в splits/:")
for f in split_files:
    lines = [l for l in f.read_text().splitlines() if l.strip()]
    print(f"  {f.name:<25s}  —  {len(lines)} записей")

Файлы в splits/:
  test_X.txt                 —  359 записей
  train_X.txt                —  694 записей
  val_X.txt                  —  328 записей


In [11]:
def load_split(name):
    """Загружает список id патчей из файла splits/{name}_X.txt."""
    path = SPLITS_PATH / f"{name}_X.txt"
    return [line.strip() for line in path.read_text().splitlines() if line.strip()]

train_ids = load_split("train")
val_ids   = load_split("val")
test_ids  = load_split("test")

print(f"Train: {len(train_ids):4d} патчей")
print(f"Val:   {len(val_ids):4d} патчей")
print(f"Test:  {len(test_ids):4d} патчей")
print(f"Итого: {len(train_ids)+len(val_ids)+len(test_ids):4d} патчей")

Train:  694 патчей
Val:    328 патчей
Test:   359 патчей
Итого: 1381 патчей


In [12]:
print("Примеры id из train:")
for pid in train_ids[:8]:
    print(f"  {pid}")

Примеры id из train:
  1-12-19_48MYU_0
  1-12-19_48MYU_1
  1-12-19_48MYU_2
  1-12-19_48MYU_3
  11-1-19_19QDA_0
  11-1-19_19QDA_1
  11-1-19_19QDA_2
  11-1-19_19QDA_3


## 5. Функция получения путей к файлам патча

По id патча нужно найти `.tif` (снимок) и `_cl.tif` (маска). Напишем вспомогательную функцию.

In [13]:
def patch_paths(patch_id):
    """
    По id патча ('1-12-19_48MYU_0') возвращает пути:
      - к файлу снимка (.tif)
      - к файлу маски классов (_cl.tif)
    """
    parts      = patch_id.rsplit("_", 1)
    scene_name = "S2_" + parts[0]           # 'S2_1-12-19_48MYU'
    patch_idx  = parts[1]                   # '0'
    base = PATCHES_PATH / scene_name / f"{scene_name}_{patch_idx}"
    img_path = base.with_suffix(".tif")
    lbl_path = Path(str(base) + "_cl.tif")
    return img_path, lbl_path

In [14]:
print("Проверка существования файлов (первые 5 патчей train):")
for pid in train_ids[:5]:
    ip, lp = patch_paths(pid)
    ok_img = "✓" if ip.exists() else "✗"
    ok_lbl = "✓" if lp.exists() else "✗"
    print(f"  {pid:<25s}  снимок {ok_img}  маска {ok_lbl}")

Проверка существования файлов (первые 5 патчей train):
  1-12-19_48MYU_0            снимок ✓  маска ✓
  1-12-19_48MYU_1            снимок ✓  маска ✓
  1-12-19_48MYU_2            снимок ✓  маска ✓
  1-12-19_48MYU_3            снимок ✓  маска ✓
  11-1-19_19QDA_0            снимок ✓  маска ✓


## 6. Детальный осмотр одного патча

In [15]:
pid = train_ids[0]
img_path, lbl_path = patch_paths(pid)

with rasterio.open(img_path) as src:
    img = src.read().astype(np.float32)
    crs       = src.crs
    transform = src.transform
    res       = src.res

print(f"Патч: {pid}")
print(f"\n--- Снимок ---")
print(f"  Форма массива  : {img.shape}  (каналы, H, W)")
print(f"  Тип данных     : {img.dtype}")
print(f"  Размер пикселя : {res[0]:.1f} × {res[1]:.1f} м")
print(f"  Проекция (CRS) : {crs}")

Патч: 1-12-19_48MYU_0

--- Снимок ---
  Форма массива  : (11, 256, 256)  (каналы, H, W)
  Тип данных     : float32
  Размер пикселя : 10.0 × 10.0 м
  Проекция (CRS) : EPSG:32748


In [16]:
print("Диапазон значений по каналам:")
print(f"  {'Канал':<6} {'λ нм':<6} {'min':>8} {'max':>8} {'mean':>10}")
print("  " + "-" * 44)
for i, (name, wl) in enumerate(zip(BAND_NAMES, BAND_WL)):
    band = img[i]
    print(f"  {name:<6} {wl:<6} {band.min():>8.1f} {band.max():>8.1f} {band.mean():>10.1f}")

Диапазон значений по каналам:
  Канал  λ нм        min      max       mean
  --------------------------------------------
  B01    443         0.1      0.1        0.1
  B02    490         0.1      0.2        0.1
  B03    560         0.1      0.2        0.1
  B04    665         0.0      0.3        0.1
  B05    705         0.0      0.2        0.0
  B06    740         0.0      0.2        0.0
  B07    783         0.0      0.2        0.0
  B08    842         0.0      0.3        0.0
  B8A    865         0.0      0.2        0.0
  B11    1610        0.0      0.2        0.0
  B12    2190        0.0      0.1        0.0


In [17]:
with rasterio.open(lbl_path) as src:
    lbl = src.read(1).astype(np.uint8)

print(f"--- Маска ---")
print(f"  Форма              : {lbl.shape}")
print(f"  Уникальные значения: {sorted(np.unique(lbl))}")
print()
print("  Распределение пикселей:")
for cls_id, cnt in sorted(Counter(lbl.ravel().tolist()).items()):
    name = CLASS_NAMES.get(cls_id, "?")
    pct  = 100 * cnt / lbl.size
    print(f"    [{cls_id:2d}] {name:<35s} {cnt:6d} пкс ({pct:.1f}%)")

--- Маска ---
  Форма              : (256, 256)
  Уникальные значения: [np.uint8(0), np.uint8(5), np.uint8(7), np.uint8(14)]

  Распределение пикселей:
    [ 0] Нет данных                           65007 пкс (99.2%)
    [ 5] Корабль                                 96 пкс (0.1%)
    [ 7] Морская вода                            83 пкс (0.1%)
    [14] ?                                      350 пкс (0.5%)


In [18]:
h_img, w_img = img.shape[1], img.shape[2]
h_lbl, w_lbl = lbl.shape

print(f"Снимок : {h_img} × {w_img} пикселей")
print(f"Маска  : {h_lbl} × {w_lbl} пикселей")
print(f"Совпадают: {h_img == h_lbl and w_img == w_lbl}")

Снимок : 256 × 256 пикселей
Маска  : 256 × 256 пикселей
Совпадают: True


## 7. Проверка нескольких патчей — размеры и типы данных

Убеждаемся, что все патчи имеют одинаковую структуру (11 каналов, одинаковые размеры снимка и маски).

In [19]:
sample_shapes = []
sample_ranges = []

for pid in train_ids[:30]:
    ip, lp = patch_paths(pid)
    try:
        with rasterio.open(ip) as s:
            arr = s.read().astype(np.float32)
        sample_shapes.append(arr.shape)
        sample_ranges.append((arr.min(), arr.max()))
    except Exception as e:
        print(f"  Ошибка {pid}: {e}")

unique_shapes = Counter(sample_shapes)
print("Формы снимков (первые 30 патчей):")
for shape, cnt in unique_shapes.most_common():
    print(f"  {str(shape):<20s}: {cnt} патчей")

all_mins = [r[0] for r in sample_ranges]
all_maxs = [r[1] for r in sample_ranges]
print(f"\nДиапазон значений:  min={min(all_mins):.1f}  max={max(all_maxs):.1f}")

Формы снимков (первые 30 патчей):
  (11, 256, 256)      : 30 патчей

Диапазон значений:  min=0.0  max=1.4


In [20]:
print("=" * 50)
print("ИТОГОВАЯ СВОДКА")
print("=" * 50)
print(f"  Сцен          : {len(scenes)}")
print(f"  Train патчей  : {len(train_ids)}")
print(f"  Val патчей    : {len(val_ids)}")
print(f"  Test патчей   : {len(test_ids)}")
print(f"  Каналов       : {len(BAND_NAMES)}")
print(f"  Целевых классов: {len(TARGET_NAMES)}")
print()
print("Далее: 02_visualization.ipynb")

ИТОГОВАЯ СВОДКА
  Сцен          : 63
  Train патчей  : 694
  Val патчей    : 328
  Test патчей   : 359
  Каналов       : 11
  Целевых классов: 4

Далее: 02_visualization.ipynb
